# UST 2y / 10y RV Score

A daily call on the direction of the **2-year** and **10-year** US Treasury yield over the next **6 months** (126 trading days):

| Output | Values | Meaning |
|---|---|---|
| **RV 2y** | +1 / 0 / -1 | +1 = buy: the 2y yield is expected to be **lower** in 6 months; -1 = sell: expected **higher**; 0 = no view |
| **RV 10y** | +1 / 0 / -1 | the same for the 10y |

Run it top to bottom on any day with no edits (Run All). It downloads new data, rebuilds every input as it was published on each date, scores 46 factors, makes the call, and saves a dated snapshot to `runs/`.

## Contents

1. **Today's call**
2. **The call against the actual yield**: how the model has done, by eye
3. **What is driving today's call**
4. **Where the composite sits, and what has followed similar readings**
5. **Full scorecard**: every factor today
6. **Track record**: P&L against simple alternatives, and how much to believe it
7. **Factor health checks**
8. **Reference**: factor guide, definitions, data provenance, fair value model, data sources

## How the call is made

1. **46 factors** across eight themes (growth, labor, inflation, Fed policy and liquidity, supply, valuation, momentum and positioning, global risk), from FRED, the Treasury, the CFTC and the NY Fed. Bloomberg-only factors are flagged unavailable when no terminal is connected.
2. **Each factor becomes a z-score**: how unusual today's reading is against its own trailing 10 years (at least 5 years of history), capped at +/-3.
3. **Each z-score becomes a pressure** by applying the factor's economic sign. **Positive pressure pushes yields higher; negative pushes them lower.** Example: strong payrolls have a positive z and a + sign, so positive pressure.
4. **Factors roll up into themes, and themes into one composite per tenor**, using fixed hand-set weights. The composite is itself re-scaled to a z-score, so it is in standard deviations.
5. **The composite becomes the call** with two thresholds: a position is entered when the composite passes **+/-0.50** and closed only when it comes back inside **+/-0.25**. The gap stops the call flipping on small moves.

Pressure is in yield terms and the call is in price terms, so they have opposite signs: **a strongly negative composite (yields expected lower) gives RV +1 (buy).**

## Key terms

| Term | Meaning |
|---|---|
| **Point-in-time** | Each value is what was actually published on that date; later revisions count only from their release date. Macro data comes from ALFRED, FRED's archive of past releases. |
| **z-score** | (today's value minus its trailing 10-year average) / its trailing 10-year standard deviation. 0 = normal, +/-2 or more = extreme. |
| **Pressure** | z-score x the factor's sign: positive = pushing yields higher. |
| **Theme / composite** | A theme is a group of related factors; the composite is the weighted average of the theme scores for one tenor. |
| **IC** | Rank correlation between the composite and the yield change that actually followed over 6 months. Positive = the model points the right way. |
| **Hit rate** | Of the days the call was +1 or -1, the share where the yield then moved the right way over 6 months. |
| **Overlay P&L / Sharpe** | P&L of being long 1 unit of DV01 at +1, short at -1, flat at 0, in basis points, including carry versus fed funds and roll-down. Sharpe = annualized return per unit of volatility. |

## What to keep in mind

- **2y**: the stronger signal. It has a clear link to subsequent yield changes and has beaten simply staying long, including after netting out what the forward curve already priced (section 6). A simple 12-month momentum rule has done about as well in P&L.
- **10y**: weak. Its record is close to always-long and it has spent much of the period since 2023 long while yields rose. Treat the 10y call as low conviction.
- **It is a backtest**: each day's call uses only data available that day, but the factor signs and weights were set by people who knew this history, and some early inputs (before about 2016) are approximations (section 8.4). There are only about 35-40 independent 6-month periods since 2009, so every statistic has wide error bars.
- The model's design, tests and the alternatives that were tried and rejected are documented in the project's plan and memo files, not in this notebook.

### Setup and data refresh

Loads the project code and runs the whole pipeline once; everything below only displays the stored `result`. Downloads happen at most once per day per series (later runs that day use the local cache). Takes about 1 minute; expect warnings about Bloomberg tickers on a machine without a terminal.

In [ ]:
import logging
import plotly.io as pio
from IPython.display import HTML, display

from ust_rv import report
from ust_rv.config import composite_model
from ust_rv.runner import run

logging.basicConfig(level=logging.ERROR, format='%(levelname)s %(name)s: %(message)s')
pio.renderers.default = 'plotly_mimetype+notebook'

result = run(with_backtest=True)

## 1. Today's call

- **Run / market data through**: when the notebook ran, and the last date with an actual Treasury yield (FRED publishes with about a one-day lag). Every chart's "as of" date is this market date.
- **Curve read**: the two calls read together. RV 2y more bullish than RV 10y = *steepener* (long 2y versus 10y); less bullish = *flattener*; equal = *neutral*. Display only, not a separate signal.
- **RV 2y / RV 10y** and the **composite z** behind each: negative = factors point to lower yields.
- **next: ... needs ...**: the nearest threshold that would change the call and how far the composite must move to reach it. Example: "exit to 0 (-0.25), needs +0.23" means the composite must rise 0.23 standard deviations before a buy goes back to neutral.
- **Since last run**: the previous call and composite, and any factor whose z moved more than 0.5.
- **Data status**: factors that are `ok`, `stale` (download failed, cached data used), `unavailable` (Bloomberg only) or `error` (listed in red), and any factor whose data is older than its normal release lag.

In [ ]:
display(HTML(report.header_html(result)))

## 2. The call against the actual yield

The quickest way to judge the model. One chart per tenor: the actual Treasury yield since the first call (2005), shaded by the call in force each day:

- **green** = RV +1 (buy): yield expected **lower** over the next 6 months
- **red** = RV -1 (sell): yield expected **higher**
- **unshaded** = neutral

The legend gives the share of days in each state. A good call is a green stretch followed by a falling yield, or a red stretch followed by a rising one, judged over the following 6 months. Section 6 puts numbers on what these charts show.

In [ ]:
for tenor in ('2y', '10y'):
    report.fig_yield_signal(result, tenor).show()

## 3. What is driving today's call

### By theme

Today's composite z split into its eight themes. Each bar is one theme's exact contribution: its weight share x its z-score, scaled the same way as the composite. **Red bars push toward higher yields, green toward lower.** The label gives the theme's weight for this tenor; weights differ by tenor because the 2y is anchored by Fed policy while the 10y carries more term premium and supply risk.

The **black diamond** on each row is where that theme stood **3 months ago** (63 trading days, with the data available then). The distance from the diamond to the end of the bar is how much the theme has moved in 3 months; a diamond on the other side of zero means it has flipped direction.

The note under each chart completes the arithmetic: theme bars plus one constant (the composite's 10-year average, subtracted when the composite z is formed) equal today's composite z exactly. It also gives the composite z 3 months ago.

In [ ]:
for tenor in ('2y', '10y'):
    report.fig_theme_breakdown(result, tenor).show()

### By factor

The same view one level down: each factor inside the themes above. **Red bars push toward higher yields, green toward lower.** Factors with an **orange outline** are flagged WARN by the health check (section 7): they have recently pointed the wrong way, so a call resting heavily on them deserves extra scepticism.

The **black diamond** on each row is where that factor's contribution stood **3 months ago** (63 trading days, calculated the same way with the data available then). The distance from the diamond to the end of the bar is how much the factor has moved in 3 months: a diamond far from its bar marks a factor that has changed the call; a diamond on the bar is a factor that has been steady. A diamond on the other side of zero means the factor has flipped direction.

The note under each chart completes the arithmetic, and gives the composite z today and 3 months ago: factor bars plus two constant terms (the themes' and the composite's 10-year averages, subtracted when the z-scores are formed) equal today's composite z exactly. It also gives the WARN factors' share of the total.

In [ ]:
for tenor in ('2y', '10y'):
    report.fig_factor_breakdown(result, tenor).show()

### Themes over the last 3 years

Top: the composite z. Bottom: each theme's z-score by week, **red = pushing yields higher, blue = lower**, darker = stronger. Read across a row to see whether a theme's push is new or long-standing; read down a column to see which themes drove the composite that week.

In [ ]:
for tenor in ('2y', '10y'):
    report.fig_theme_heatmap(result, tenor).show()

## 4. Where the composite sits, and what has followed similar readings

### Composite history

- **Composite z** (left axis, blue for 2y, purple for 10y) and the **actual yield** (right axis, dark line).
- **Shading** = the call in force (green buy, red sell).
- **Dashed lines** = entry thresholds (+/-0.50): crossing above +0.50 turns the call to sell, below -0.50 to buy. **Dotted lines** = exit band (+/-0.25): a position is held until the composite comes back inside it.

In [ ]:
for tenor in ('2y', '10y'):
    report.fig_composite(result, tenor).show()

### Where today's composite sits on the decision scale

One row per tenor, read like a gauge. The composite z runs left (factors point to lower yields) to right (higher yields), divided into five zones:

| Zone | What happens there |
|---|---|
| **BUY (+1)**, below -0.50 | The call becomes or stays +1. |
| **keeps current call**, -0.50 to -0.25 | No new position is opened, but an existing buy is kept. A neutral call stays neutral. |
| **NEUTRAL (0)**, -0.25 to +0.25 | Any position is closed; the call is 0. |
| **keeps current call**, +0.25 to +0.50 | An existing sell is kept; a neutral call stays neutral. |
| **SELL (-1)**, above +0.50 | The call becomes or stays -1. |

The two "keeps current call" zones are what stop the call flipping back and forth when the composite hovers near a threshold: it takes a move all the way back into the neutral zone to close a position.

The **dot** is today's composite (coloured by the current call: green buy, red sell, grey neutral); the **diamond and arrow** show where it was 3 months ago and which way it has moved. The line above each row gives the call, the composite now and 3 months ago, and the nearest threshold that would change the call.

In [ ]:
report.fig_band_position(result).show()

### What has happened after readings like today's

Every day since 2009 is grouped by its composite reading into buckets of equal width, from **factors pointing strongly to lower yields** (left) to **strongly higher** (right). Buckets are 0.25 wide between -1 and +1, where most days fall, and 0.5 wide out to +/-2, with one open bucket at each end. The edges line up with the model's thresholds (+/-0.25 exit, +/-0.50 entry). Each label gives the bucket's range and its number of days (n). For each bucket, the chart shows what the yield actually did over the following 6 months:

- **box** = the **middle half** of outcomes (25th to 75th percentile)
- **whiskers** = the typical range (out to 1.5 times the box height beyond the box; rarer extremes are not drawn)
- **red line and markers** = the **median** outcome in each bucket, joined left to right
- **gold box** = the bucket that contains **today's** reading
- **faded box** = fewer than 120 days in the bucket (less than one independent 6-month outcome): read with caution

The sentence under each chart states what followed readings like today's: the median change, the middle-half range, and how often yields rose.

**How to read it**: if the model works, the red line rises from left to right (low readings followed by falling yields, high readings by rising yields). The boxes and whiskers show how wide the outcomes are around that median: even in the best buckets, individual 6-month outcomes vary a lot. Compare the two tenors: a steady climb means the reading is informative, a wandering line means it tells you little. The outermost buckets come from a handful of episodes (mainly the 2008-09 and 2020 crises), so their medians are much less reliable than those in the middle.

In [ ]:
for tenor in ('2y', '10y'):
    report.fig_conditional_outcomes(result, tenor).show()

### The yield path after each new call

Every time the call switched into a buy (left) or a sell (right) since 2005: how the yield moved over the next 6 months, in bp from the entry day. The thick line is the median path; the titles give the number of entries. The **label in each panel** gives the share of completed calls where the yield had moved the right way after 6 months (lower after a buy, higher after a sell), the number of completed calls, and the median 6-month move.

After a buy you want the lines to drift down, after a sell up. The median shows the typical speed of the move; the worst lines show what a position had to sit through. With only a handful of entries per side, a single episode can dominate.

In [ ]:
for tenor in ('2y', '10y'):
    report.fig_entry_paths(result, tenor).show()

## 5. Full scorecard

Every factor today, grouped by theme with a bold subtotal row per theme. Within a theme, rows are sorted by how much they matter today.

| Column | Meaning |
|---|---|
| **ID, Factor** | The factor's ID (A1, B2, ...) and name; units are in the name (%, bp, k = thousands). Section 8.1 explains each factor. |
| **Today's value** | Today's reading after its transform (for example the 3-month annualized % change in core PCE). |
| **Data released / Days since release** | When the data behind the value was published, and how many days ago. Monthly data is normally 15-45 days old. |
| **Z-score vs 10y history** | How unusual today's value is against the factor's own last 10 years (0 = normal, +/-2 = extreme). |
| **Z-score change, last month** | How much that z-score has moved over the last 21 trading days. |
| **Pressure on yields** | The z-score with the factor's economic sign applied. **Red = pushing yields higher, green = lower**; stronger shading = larger. |
| **Weight in theme, 2y / 10y** | The factor's weight within its theme for each tenor (0 = not used for that tenor). |
| **Share of theme push, 2y / 10y** | How much of its theme's total push this factor provides. The shares within a theme add up to the bold theme row. |
| **Sign check, 2y / 10y** | `PASS` = over the last 5 years the factor has lined up with what yields actually did; `WARN` = it has pointed the wrong way. Signs are never flipped automatically. |
| **Data source** | `alfred:` = FRED's archive of past releases (as published on each date), `fred:` = FRED daily market data, `public:` = Treasury / CFTC / NY Fed, `model:` = computed here; `a - b` or `a / b` = a spread or ratio of two series. |
| **Data status** | `ok`, `stale` (download failed, cached data used), `unavailable` (Bloomberg only) or `error`. |

Valuation factors are contrarian by construction: a high term premium or a cheap fair-value residual is green (pushing yields lower), because cheap bonds tend to rally.

In [ ]:
report.style_scorecard(report.scorecard_table(result))

### Theme numbers

| Column | Meaning |
|---|---|
| **Average factor pressure** | The weighted average pressure of the theme's factors (the bold theme row in the scorecard). |
| **Theme z-score** | That pressure re-scaled against its own 10 years, so themes with many factors do not dominate. |
| **Theme weight** | The theme's weight in the composite for that tenor. |
| **Share of composite** | Theme weight x theme z-score / sum of weights: the theme's share of the composite before its final re-scaling (section 3 shows the same split in composite z units). |

In [ ]:
display(report.theme_numbers(result))

## 6. Track record

Everything here is a backtest of the live model from 2005: each day's call uses only data available that day.

### Did following the model make money, and more than the simple alternatives?

| Line | Strategy |
|---|---|
| **Following the model** (thick, blue for 2y, purple for 10y) | Long at +1, short at -1, flat at 0. |
| **Always long** (grey) | Long all the time. The benchmark any duration call must beat: it earns carry and benefited from the long fall in yields. |
| **Momentum only** (gold) | Long when yields have been falling over the past year, short when rising. A one-line trend rule. |

**Units**: cumulative P&L in basis points for a position of 1 unit of DV01, including the yield move, carry versus fed funds and roll-down. The legend gives each total since 2005. The bottom panel is the model's **drawdown**: how far its P&L sat below its previous high.

In [ ]:
for tenor in ('2y', '10y'):
    report.fig_pnl_attribution(result, tenor).show()

### Where each strategy's P&L came from

Average P&L per year since 2005 (bp per 1 unit of DV01):

| Column | Meaning |
|---|---|
| **calling the move** | Gains from being on the right side of yield moves: the part that reflects forecasting skill. |
| **carry** | Yield earned versus financing at fed funds while a position is held; longs earn it on a positive spread, shorts pay it. |
| **roll-down** | Gain from the bond's yield falling as it ages along an upward-sloping curve; same sign logic as carry. |
| **total, Sharpe** | Sum per year, and the annualized Sharpe ratio. |

A model with real skill earns most of its money in "calling the move". Always-long earns mainly carry and roll.

In [ ]:
display(report.pnl_sources_table(result))

### Performance table

The live model and the two benchmarks over three windows: **2005+** (the full backtest), **2009+**, and **PIT>=90%** (only days where at least 90% of factor data is strictly as-published, about 2016 on: the cleanest test).

| Column | Meaning |
|---|---|
| **IC, IC CI** | Rank correlation with the realized 6-month change, with 95% interval. An interval that includes 0 means skill is not established. |
| **HAC t** | t-statistic corrected for overlapping 6-month periods. Read with care: the composite and the target are both very persistent; the "how much to believe" chart below is more robust. |
| **hit rate, hit CI** | Share of +1/-1 days where the yield then moved the right way over 6 months. Positions typically last 2-3 months, so the P&L columns are the more direct measure. |
| **Sharpe, Sharpe CI** | Annualized Sharpe of the overlay P&L, with 95% interval. |
| **bp/yr, max adverse bp** | Average P&L per year; the largest 6-month yield move against an open position. |
| **neutral, avg hold d, turnover/yr** | Share of days at 0; average position length in trading days; call changes per year. |

In [ ]:
for tenor, bt in result.backtests.items():
    print(tenor)
    display(report.backtest_metrics_frame(bt, composite_model(tenor)))

### Is the skill more than what the curve already prices?

The forward curve already prices an expected path; when cuts are priced, part of a fall in yields is "in the price". This table scores the composite against the raw 6-month change and against the move **beyond** today's 6-month forward yield (from the FRED Treasury curve; par yields used as zero yields, an approximation of a few bp). If the second IC stays close to the first, the model is predicting moves the market had not priced. The P&L above already includes carry and roll, so it is unaffected.

In [ ]:
display(report.forward_ic_table(result))

### When has the composite worked?

Top: IC over a rolling 3-year window. Bottom: IC within each calendar year. Green bars = the composite pointed the right way that year, red = the wrong way. A reliable signal is positive most of the time, not just on average; red years show the conditions under which it fails.

In [ ]:
for tenor in ('2y', '10y'):
    report.fig_rolling_ic(result, tenor).show()

### How much to believe the IC

Two independent checks per tenor (2009 onward):

- **Left, placebo**: the same 46-factor machinery rebuilt 300 times with **random factor signs**. The grey histogram is the IC those random composites achieve; the coloured line is the real composite. Far to the right = unlikely to be luck from the structure alone.
- **Right, independent windows**: the IC computed only on non-overlapping 6-month periods (about 34 points each), once for each of the 126 possible start days. This avoids the overlap problem entirely; a distribution well above zero is robust.

Takes about 25 seconds.

In [ ]:
for tenor in ('2y', '10y'):
    report.fig_credibility(result, tenor).show()

## 7. Factor health checks

Each factor has an assumed economic sign. These checks show whether that has held. They inform judgment; signs are never changed automatically.

### Factors currently flagged WARN

Factors whose rank correlation with the realized 6-month change over the last 5 years is below -0.10 for at least one tenor (`rho_2y` / `rho_10y`). Many WARNs are expected after unusual periods such as the zero-rate years and the 2022 hiking cycle. A factor that stays WARN for years is a candidate for review.

In [ ]:
display(report.warn_list(result))

### Sign stability over time

Rows are factors, columns are month-ends since 2005; colour is the trailing 5-year rank correlation between the factor's pressure and the realized 6-month change. **Blue** = behaved as signed, **red** = contradicted its sign, blank = not enough history. Persistently red rows suggest a mis-specified factor; columns where many rows turn red at once show a regime where usual relationships broke down.

In [ ]:
for tenor in ('2y', '10y'):
    report.fig_sign_stability(result, report.sign_stability(result, tenor), tenor).show()

### Each factor's track record

One row per factor: rank correlation of its pressure with the realized 6-month change. **Coloured bar = since 2005** (green = behaves as signed, red = wrong way); **grey bar = trailing 5 years** (the WARN measure; the dotted orange line is the -0.10 threshold). Weight within its theme is in the label. Factors work together: removing any single weak factor changes the composite very little, and dropping factors on their track record has not improved results.

In [ ]:
for tenor in ('2y', '10y'):
    report.fig_factor_ic(result, tenor).show()

### Which factors say the same thing

Correlation between factor pressures since 2009, reordered so similar factors sit together. **Dark blocks along the diagonal are groups carrying the same information**, which get more total weight than their theme weight suggests. The growth and labor indicators cluster as expected (the business cycle); unexpected high correlations across themes are worth a look.

In [ ]:
for tenor in ('2y', '10y'):
    report.fig_factor_correlation(result, tenor).show()

## 8. Reference

### 8.1 Factor guide: what each factor is and why it matters

This guide explains every factor in plain language, theme by theme: what it measures, why it moves Treasury yields, how to read high and low values, and how it relates to the other factors. The factor IDs (A1, B2, ...) match the scorecard in section 1. The table after the guide gives the compact technical definition of each one.

**Reading the weights.** Each factor shows its weight within its theme for the 2y and the 10y (0 = not used for that tenor), and its sign. A **+** factor pushes yields higher when its value is high; a **-** factor pushes yields lower when its value is high. Each theme also has a weight in the composite, given at the top of its section.

**Why 2y and 10y differ.** The 2y yield is mostly the market's forecast of the fed funds rate over the next two years, so it reacts to anything that changes the Fed's next moves: inflation, the labor market, financial conditions. The 10y adds two things the 2y barely has: a long-run view of growth and inflation, and a term premium (the extra yield investors demand for tying money up for 10 years). Supply, global yields and valuation therefore matter more for the 10y.

#### Theme A: Growth and activity (composite weight 2y 1.0, 10y 1.0)

Faster growth raises the odds of Fed tightening, raises the neutral level of rates, and draws money out of bonds into risk assets, so strong growth pushes yields higher. The theme mixes a broad nowcast, a broad activity index and three specific sectors so that no single noisy release dominates.

**A1. GDPNow nowcast** (2y 1.0, 10y 1.0, sign +)
- *What it is*: the Atlanta Fed's running model estimate of real GDP growth for the current quarter, annualized, updated several times a month as new data arrive.
- *Why it matters*: it is the timeliest single read on whether the economy is running hot or cold, months before the official GDP release.
- *How to read it*: around 2% is roughly trend growth; well above 2% pushes yields up, near or below 0% pushes them down.
- *Caveat*: early in a quarter it rests on few data points and can swing. History before 2016 is approximated from final nowcasts.

**A2. CFNAI 3-month average** (2y 1.0, 10y 1.0, sign +)
- *What it is*: the Chicago Fed National Activity Index, a weighted average of 85 monthly indicators across production, employment, consumption and housing, averaged over 3 months.
- *Why it matters*: it is the broadest single gauge of activity, so it confirms or challenges what the narrower series say.
- *How to read it*: 0 = trend growth. Positive = above trend. Below about -0.7 has historically coincided with recessions.

**A3. Industrial production, 6-month annualized change** (2y 1.0, 10y 1.0, sign +)
- *What it is*: the Fed's index of factory, mining and utility output.
- *Why it matters*: manufacturing is the most cyclical part of the economy and turns early.
- *How to read it*: positive and rising = expanding output, pushing yields up; negative = contraction.

**A4. Retail sales, 3-month over 3-month annualized** (2y 1.0, 10y 0.5, sign +)
- *What it is*: the Census advance estimate of retail and food-service sales, comparing the last 3 months with the 3 before.
- *Why it matters*: consumer spending is about two-thirds of GDP, and strong spending keeps the Fed cautious about cutting.
- *How to read it*: it is in nominal terms, so it includes inflation; strong readings with high inflation are less informative about real growth. Weighted more for the 2y because it feeds near-term Fed decisions.

**A5. Building permits, 12-month change** (2y 0.5, 10y 1.0, sign +)
- *What it is*: new private housing units authorized by permits.
- *Why it matters*: housing is the sector most sensitive to mortgage rates and leads the broader cycle; falling permits often flag a slowdown before it shows up elsewhere.
- *How to read it*: positive = housing expanding. Weighted more for the 10y because mortgage rates key off longer yields.

**A6. Economic surprise index** (2y 1.0, 10y 1.0, sign +; Bloomberg only)
- *What it is*: the Citi Economic Surprise Index, which tracks whether data releases have beaten or missed economists' forecasts.
- *Why it matters*: markets move on surprises, not on levels; persistent upside surprises force forecasts and yields higher.
- *Status*: unavailable without a Bloomberg terminal, so it currently drops out and the theme re-weights over the other five.

#### Theme B: Labor market (composite weight 2y 1.25, 10y 1.0)

Maximum employment is half of the Fed's mandate, so the labor market drives Fed expectations and hence the 2y. The theme covers hiring (payrolls), layoffs (claims), slack (Sahm rule), tightness (openings per unemployed) and pay (wages). Weighted more for the 2y because labor data move the expected path of the policy rate most directly.

**B1. Payrolls, 3-month average change** (2y 1.0, 10y 1.0, sign +)
- *What it is*: the monthly change in nonfarm payroll jobs from the BLS establishment survey, averaged over 3 months, using the figures as revised up to each date.
- *Why it matters*: the most market-moving monthly release; sustained strong hiring means the Fed is in no hurry to cut.
- *How to read it*: the pace needed to keep unemployment steady depends on labor force growth (estimates have ranged from well under 100k to about 150k a month, and fell with lower immigration). Well above that is a hot market; near zero or negative signals weakness.

**B2. Sahm rule gap** (2y 1.0, 10y 1.0, sign -)
- *What it is*: the 3-month average unemployment rate minus its low over the prior 12 months, in percentage points.
- *Why it matters*: a rising unemployment rate is the classic recession signal, and recessions bring Fed cuts and lower yields.
- *How to read it*: near 0 = unemployment not rising. A reading of 0.5 or more has historically marked the start of a recession. Because the sign is -, a rising gap pushes yields lower.

**B3. Initial jobless claims, 4-week average versus a year earlier** (2y 1.0, 10y 1.0, sign -)
- *What it is*: weekly new claims for unemployment insurance, smoothed over 4 weeks and compared with the same period a year earlier.
- *Why it matters*: it is the most timely labor indicator (weekly), and the first to show layoffs picking up.
- *How to read it*: claims rising 10-20% or more year on year is a warning sign; falling claims push yields up (sign -).

**B4. Job openings per unemployed person** (2y 1.0, 10y 0.5, sign +)
- *What it is*: JOLTS job openings divided by the number of unemployed people.
- *Why it matters*: it measures labor-market tightness, which drives wage pressure and the Fed's view of how much slack exists.
- *How to read it*: around 1 = balanced; well above 1 (it peaked near 2 in 2022) = very tight, pushing yields up. Released with a lag of about 5 weeks.

**B5. Average hourly earnings, 3-month annualized** (2y 1.0, 10y 1.0, sign +)
- *What it is*: the change in average hourly pay for private-sector workers over 3 months, annualized.
- *Why it matters*: wages are the main cost in services, so fast wage growth keeps services inflation sticky.
- *How to read it*: roughly 3-3.5% is consistent with 2% inflation given normal productivity growth; well above that pushes yields up. Can be distorted when low-paid jobs are lost or added in bulk (composition effects).

#### Theme C: Inflation (composite weight 2y 1.5, 10y 1.25)

Price stability is the other half of the Fed's mandate, and inflation erodes the real value of every coupon, so it is the most heavily weighted theme for both tenors. The factors are chosen to cover different angles without double-counting the same data: the *current rate* comes from PCE (C1, C3), *acceleration* from CPI (C2), *expectations* from markets (C4) and households (C5), and an *early input* from oil (C6). Headline inflation and the level of CPI are deliberately left out to avoid near-duplicates.

**C1. Core PCE, 3-month annualized** (2y 1.0, 10y 1.0, sign +)
- *What it is*: the core personal consumption expenditures price index (excluding food and energy), the measure the Fed's 2% target refers to, as its 3-month annualized change.
- *Why it matters*: it is the number the Fed actually targets, and the 3-month rate shows where inflation is heading now rather than where it was over the past year.
- *How to read it*: 2% = on target. Persistently above 2.5-3% keeps the Fed restrictive and pushes yields up.

**C2. Core CPI acceleration** (2y 1.0, 10y 0.5, sign +)
- *What it is*: core CPI's 3-month annualized change minus its 12-month change.
- *Why it matters*: CPI is released about two weeks before PCE and moves markets more on the day. Using it as acceleration (is inflation speeding up or slowing down?) adds a different signal from C1 rather than repeating the level.
- *How to read it*: positive = recent inflation running hotter than the past year, pushing yields up; negative = disinflation underway.

**C3. Trimmed-mean PCE, 12-month** (2y 1.0, 10y 1.0, sign +)
- *What it is*: the Dallas Fed's trimmed-mean PCE inflation rate, which drops the most extreme price moves each month (both up and down) and averages the rest.
- *Why it matters*: it strips out one-off swings (used cars, airfares) and shows how broad-based inflation is.
- *How to read it*: close to core PCE when inflation is broad; when it diverges, it says a few categories are driving the headline numbers.

**C4. 5y5y forward breakeven inflation** (2y 0.5, 10y 1.0, sign +)
- *What it is*: the average inflation rate that the TIPS and nominal Treasury markets imply for the five years starting five years from now.
- *Why it matters*: it is the market's read on long-run inflation expectations, which the Fed watches for signs of de-anchoring, and it is a direct component of long-term yields. Weighted more for the 10y for that reason.
- *How to read it*: roughly 2.0-2.5% is consistent with anchored expectations; a rise above that pushes long yields up. Available from 2003.

**C5. University of Michigan 1-year inflation expectations** (2y 1.0, 10y 0.5, sign +)
- *What it is*: households' median expected inflation over the next 12 months, from the monthly UMich survey.
- *Why it matters*: household expectations can feed into wage demands and pricing, and the Fed cites them as a risk to its credibility.
- *How to read it*: tends to track gasoline prices and recent inflation; readings well above 3% concern the Fed. FRED posts it with a one-month delay.

**C6. WTI oil, 3-month change** (2y 1.0, 10y 0.5, sign +)
- *What it is*: the percentage change in the WTI crude oil price over the last 3 months.
- *Why it matters*: oil feeds headline inflation quickly through gasoline and transport costs, and moves inflation expectations (C4, C5) before it shows up in the official data.
- *How to read it*: a sharp rally pushes yields up. A sharp fall can reflect a demand shock (bearish growth), which also argues for lower yields.

#### Theme D: Fed policy and liquidity (composite weight 2y 1.5, 10y 1.0)

These factors measure how tight policy and financial conditions already are, and how much easing or tightening the market expects. Several are deliberately read in reverse: tight policy today slows the economy and pulls yields down later. Weighted most heavily for the 2y, which is anchored to the policy rate.

**D1. Real policy rate** (2y 1.0, 10y 0.5, sign -)
- *What it is*: the effective fed funds rate minus 12-month core PCE inflation.
- *Why it matters*: what restrains the economy is the policy rate after inflation. A high real rate means policy is restrictive, which slows growth and eventually brings cuts.
- *How to read it*: Fed and academic estimates of the neutral real rate are mostly in the 0.5-1.5% range; well above that is restrictive and pushes yields lower ahead (sign -).

**D2. Priced policy path, 1 year** (2y 1.0, 10y 0.5, sign -; Bloomberg only)
- *What it is*: how far the market expects the policy rate to move over the next year: the 1-year OIS rate minus fed funds. **Bloomberg only**: without a terminal D2 is unavailable and the Fed theme uses D1, D3, D4 and D5. (The free alternative, the 1-year bill minus fed funds, is almost the same spread as F5, correlation about 0.9, so it would count one signal twice.)
- *Why it matters*: it is read as contrarian. When the market already prices deep cuts, much of the good news for bonds is in the price and the risk skews toward yields rising if the cuts disappoint.
- *How to read it*: negative = cuts priced, which this factor turns into pressure for higher yields. It goes wrong in genuine easing cycles, when the cuts do arrive; the sign check flags when that is happening.

**D3. Fed balance sheet, 6-month change** (2y 0.5, 10y 1.0, sign -)
- *What it is*: the percentage change in the Fed's total assets over 26 weeks.
- *Why it matters*: when the Fed buys bonds (QE) it removes duration from the market and lowers term premium; when it shrinks its balance sheet (QT) the market must absorb more. Weighted more for the 10y, where QE operates.
- *How to read it*: positive = balance sheet growing, pushing yields lower; negative = QT, pushing yields higher.

**D4. Bank reserves, 3-month change** (2y 0.5, 10y 1.0, sign -)
- *What it is*: the percentage change over 13 weeks in reserves that banks hold at the Fed.
- *Why it matters*: reserves are the system's liquidity. Falling reserves can tighten funding markets and force balance sheet strain; rising reserves support demand for Treasuries.
- *How to read it*: rising reserves push yields lower. Also moved by the Treasury's cash balance and the Fed's reverse repo facility, not only by QE or QT.

**D5. Chicago Fed financial conditions index (NFCI)** (2y 1.0, 10y 1.0, sign -)
- *What it is*: a weekly index built from 105 measures of risk, credit and leverage across money markets, debt and equity markets and the banking system.
- *Why it matters*: tight financial conditions slow the economy as if the Fed had hiked; very loose conditions make the Fed reluctant to ease.
- *How to read it*: 0 = average; positive = tighter than average, pushing yields lower ahead; negative = looser, pushing yields higher. The whole history is revised weekly; the model uses each week's value as it was published.

#### Theme E: Supply and fiscal (composite weight 2y 0.25, 10y 1.0)

Treasury yields also depend on how much debt the market has to absorb and how strong the demand for it is. These effects show up mainly in long maturities, through the term premium, so the theme carries full weight for the 10y and only a small weight for the 2y.

**E1. Federal deficit, % of GDP (trailing 12 months)** (2y 0, 10y 1.0, sign +)
- *What it is*: the federal budget deficit over the last 12 months from the Monthly Treasury Statement, divided by nominal GDP.
- *Why it matters*: bigger deficits must be financed with more Treasury issuance, which requires higher yields to attract buyers over time.
- *How to read it*: deficits of 6% of GDP or more outside a recession are historically unusual and push long yields up; the z-score compares today with the trailing 10 years.

**E2. 10-year auction bid-to-cover, last 3 auctions** (2y 0, 10y 1.0, sign -)
- *What it is*: total bids received divided by the amount sold, averaged over the last three 10-year note auctions (including reopenings).
- *Why it matters*: it is a direct, observable measure of demand for new 10-year supply.
- *How to read it*: higher = stronger demand, pushing yields lower. In recent years it has mostly ranged around 2.3-2.7; the z-score reads it against its own history.

**E3. 10-year auction indirect bidder share, last 3 auctions** (2y 0, 10y 1.0, sign -)
- *What it is*: the share of the 10-year auction awarded to indirect bidders, who bid through dealers and are mostly foreign central banks and large institutional investors.
- *Why it matters*: indirect demand is "real money" that holds the bonds, unlike dealers who must resell them. A high share means supply is going to strong hands.
- *How to read it*: higher = stronger end-investor demand, pushing yields lower. Data from 2008 only, so it entered the composite around 2013.

**E4. 2-year auction bid-to-cover, last 3 auctions** (2y 1.0, 10y 0, sign -)
- *What it is*: the same demand measure as E2, for 2-year note auctions.
- *Why it matters*: the 2y version of the auction demand signal.
- *How to read it*: higher = stronger demand, pushing 2y yields lower.

**E5. Coupon issuance, 3-month total versus a year earlier** (2y 0.5, 10y 1.0, sign +)
- *What it is*: the total size of all nominal note and bond auctions over the last 3 months (excluding TIPS and floating-rate notes), compared with the same 3 months a year earlier.
- *Why it matters*: rising issuance means more duration the market must absorb.
- *How to read it*: positive = more supply, pushing yields up. It reads 0 when the Treasury keeps auction sizes unchanged, as it has recently; the z-score then reflects how that compares with history.

**E6. Primary dealer inventory, 7-11 year coupons** (2y 0, 10y 1.0, sign +)
- *What it is*: the net position (long minus short, $ millions) that primary dealers hold in Treasury notes maturing in 7 to 11 years, reported weekly to the NY Fed.
- *Why it matters*: dealers buy what end investors do not take at auction. A large inventory is supply that still has to be placed, and dealers have limited balance sheet to carry it.
- *How to read it*: a high long position pushes yields up. Data from 2013.

#### Theme F: Valuation and term premium (composite weight 2y 1.0, 10y 1.25)

These factors ask whether Treasuries are cheap or expensive relative to fundamentals, and they are mostly contrarian: when bonds are cheap (high term premium, high real yield, yield above fair value), the expected return from holding them is high and yields tend to fall afterwards. They work over months, not days.

**F1. 10-year term premium (Kim-Wright)** (2y 0, 10y 1.0, sign -)
- *What it is*: a Federal Reserve Board model estimate of the extra yield investors demand for holding a 10-year bond instead of rolling short-term bills for 10 years.
- *Why it matters*: it is the part of the 10y yield that is compensation for risk rather than expected policy rates; a high premium means bonds are cheap.
- *How to read it*: it was strongly positive before 2010, near or below zero through the QE years, and positive again since 2023. High = cheap, which this factor turns into pressure toward lower yields.

**F2. 2-year term premium (Kim-Wright)** (2y 1.0, 10y 0, sign -)
- *What it is*: the same model estimate for the 2-year maturity.
- *Why it matters*: the 2y version of the valuation signal; smaller and less volatile than the 10y premium.
- *How to read it*: high = cheap 2y, pressure toward lower 2y yields.

**F3. 10-year fair value residual** (2y 0, 10y 1.0, sign -)
- *What it is*: the actual 10y yield minus the yield predicted by a regression on fed funds, core PCE inflation, real GDP growth and the size of the Fed's balance sheet (see the fair value model above).
- *Why it matters*: it measures whether the 10y has moved away from what the macro backdrop justifies.
- *How to read it*: positive = the 10y yields more than fundamentals explain (cheap), pressure toward lower yields; negative = rich, pressure toward higher yields. Less reliable when the model's R-squared is low.

**F4. 10-year real yield (TIPS)** (2y 0, 10y 1.0, sign -)
- *What it is*: the yield on 10-year Treasury Inflation-Protected Securities, i.e. the return after inflation.
- *Why it matters*: a high real yield both restrains the economy and makes bonds attractive in real terms; both argue for lower yields ahead.
- *How to read it*: around 2% or higher is high by post-2008 standards; negative real yields (2020-21) were extremely rich.

**F5. 2-year yield minus fed funds** (2y 1.0, 10y 0, sign -)
- *What it is*: the 2-year Treasury yield minus the effective fed funds rate.
- *Why it matters*: the 2y should roughly equal the average fed funds rate expected over two years. A 2y well above fed funds prices hikes and offers extra carry; well below prices cuts.
- *How to read it*: a wide positive spread means the 2y is cheap versus funding, pressure toward lower 2y yields. Read alongside D2, which measures the priced path from a different angle.

**F6. 10-year swap spread** (2y 0, 10y 0.5, sign +; Bloomberg only)
- *What it is*: the 10-year interest rate swap rate minus the 10-year Treasury yield.
- *Why it matters*: it reflects how much balance sheet strain and supply pressure Treasuries face relative to swaps; it turned deeply negative as issuance grew and dealer balance sheets tightened.
- *Status*: unavailable without a Bloomberg terminal; its definition changed with the LIBOR-to-SOFR transition, which Phase 4 must check.

#### Theme G: Momentum and positioning (composite weight 2y 0.75, 10y 0.75)

Yields trend: moves driven by a change in the Fed cycle or inflation regime tend to persist for months. Positioning measures how crowded a trade has become, which signals the risk of a sharp reversal. The theme gets a moderate weight because trends eventually turn, and it is the theme most likely to be wrong at turning points.

**G1. 10-year momentum, 12 months to 1 month ago** (2y 0, 10y 1.0, sign +)
- *What it is*: the change in the 10y yield from 12 months ago to 1 month ago, in bp. The most recent month is skipped because very short-term moves tend to reverse.
- *Why it matters*: trend-following in bond yields has worked over long histories across markets.
- *How to read it*: positive = yields have been trending up, pressure for further increases.

**G2. 2-year momentum, 12 months to 1 month ago** (2y 1.0, 10y 0, sign +)
- *What it is*: the same measure for the 2y yield.
- *Why it matters*: 2y trends follow Fed cycles, which typically run for a year or more once started. On its own this simple signal has been competitive with the full composite in the backtest (section 7).
- *How to read it*: positive = 2y yields trending up.

**G3. 10-year yield versus its 200-day average** (2y 0, 10y 1.0, sign +)
- *What it is*: the 10y yield minus its 200-trading-day moving average, in bp.
- *Why it matters*: a faster trend signal than G1, widely watched by technical traders.
- *How to read it*: above the average = uptrend in yields, pressure for further increases.

**G4. Leveraged fund positioning in 10-year futures** (2y 0, 10y 1.0, sign +)
- *What it is*: hedge funds' and other leveraged traders' net position in 10-year (TY) plus Ultra 10-year (TN) Treasury futures, as % of open interest, from the CFTC's weekly Traders in Financial Futures report.
- *Why it matters*: read as contrarian. When leveraged funds are very short, the trade is crowded and a squeeze (forced buying back) can drive yields down sharply. Note that part of the short reflects the cash-futures basis trade rather than an outright bet.
- *How to read it*: very negative = crowded short, which this factor turns into pressure toward lower yields.

**G5. Leveraged fund positioning in 2-year futures** (2y 1.0, 10y 0, sign +)
- *What it is*: the same measure for 2-year (TU) futures.
- *Why it matters / how to read it*: as G4, for the 2y.

**G6. MOVE index** (2y 0.5, 10y 0.5, sign -; Bloomberg only)
- *What it is*: the implied volatility of Treasury options, the bond market's equivalent of the VIX.
- *Why it matters*: very high rate volatility usually comes with market stress and flight-to-quality buying of Treasuries.
- *Status*: unavailable without a Bloomberg terminal.

#### Theme H: Global and risk sentiment (composite weight 2y 0.75, 10y 1.0)

US yields do not move in isolation: global bond markets are linked through investors who compare yields across countries, and risk-off episodes send money into Treasuries as a safe asset. Weighted more for the 10y, where global capital flows matter most.

**H1. German 10-year Bund yield, 3-month change** (2y 0.5, 10y 1.0, sign +)
- *What it is*: the change in the 10-year German government bond yield over 3 months, in bp (Bloomberg daily; otherwise OECD monthly data from FRED).
- *Why it matters*: Bunds are the euro area's benchmark safe asset; when Bund yields rise, European investors have less reason to buy Treasuries, and global term premium tends to rise together.
- *How to read it*: rising Bund yields push US yields up.

**H2. Japanese 10-year JGB yield, 3-month change** (2y 0, 10y 1.0, sign +)
- *What it is*: the change in the 10-year Japanese government bond yield over 3 months, in bp.
- *Why it matters*: Japanese investors are among the largest foreign holders of Treasuries. Higher yields at home, especially after the Bank of Japan's exit from yield curve control, reduce their need to buy US bonds.
- *How to read it*: rising JGB yields push US 10y yields up. Not used for the 2y.

**H3. Broad dollar index, 3-month change** (2y 0.5, 10y 1.0, sign -)
- *What it is*: the Federal Reserve's trade-weighted index of the dollar against a broad basket of currencies, % change over 3 months.
- *Why it matters*: a strong dollar tightens financial conditions globally and lowers US import prices, both of which argue for lower yields ahead.
- *How to read it*: a rising dollar pushes yields lower.

**H4. Credit spread, 3-month change** (2y 1.0, 10y 1.0, sign -)
- *What it is*: the change over 3 months in corporate credit spreads, in bp: high-yield option-adjusted spread on Bloomberg, or Moody's Baa corporate yield minus the 10y Treasury yield otherwise.
- *Why it matters*: widening spreads signal rising default fears and risk aversion, which bring flight-to-quality buying of Treasuries and raise the odds of Fed easing.
- *How to read it*: widening (positive change) pushes yields lower; tightening pushes them higher.

**H5. VIX** (2y 1.0, 10y 1.0, sign -)
- *What it is*: the implied volatility of S&P 500 options over the next 30 days.
- *Why it matters*: high equity volatility goes with risk-off flows into Treasuries and with a more cautious Fed.
- *How to read it*: roughly 12-20 is calm; above 30 is stress, pushing yields lower. Note that in 2022 stocks and bonds sold off together, when this relationship broke down.

**H6. Copper, 6-month change** (2y 0.5, 10y 1.0, sign +)
- *What it is*: the percentage change in the IMF's global copper price over 6 months.
- *Why it matters*: copper is used across construction and manufacturing worldwide, so its price is a market-based read on global industrial demand.
- *How to read it*: rising copper signals stronger global growth, pushing yields up. Monthly data with about a 2-month publication lag.

### 8.2 Factor definitions

The compact technical definition of each factor: what the series is and who publishes it, its exact transform, and the reason for its sign. **sign**: + = a higher value pushes yields higher; - = lower. **used for**: tenors with a non-zero weight. **source used today**: the exact series behind the value on this run.

In [ ]:
report.factor_definitions(result)

### 8.3 Factor library

| Column | Meaning |
|---|---|
| **source used / primary/fallback** | The series used today, and whether it was the preferred source or its backup. |
| **status, last release, max age (d)** | Data status, the latest release date, and how old the data can normally get before counting as late. |
| **history from** | First date with a value; the z-score needs 5 more years before the factor counts in the composite. |
| **true PIT share since 2005** | Share of days since 2005 with strictly as-published data (see 8.4). |

In [ ]:
display(report.factor_library_table(result))

### 8.4 Data provenance by factor

One row per factor since 1990: **gold** = history approximated because the series' archive of past releases starts later (the earliest release, shifted by its normal publication lag; for model-based series such as the Kim-Wright term premium and NFCI these values come from later re-estimates); **green** = true point-in-time history; **black tick** = when the factor starts counting in the composite (5 years of history); **empty row** = unavailable on this machine (Bloomberg only). The dotted line marks 2005, the start of the backtest.

In [ ]:
report.fig_provenance(result).show()

### 8.5 Fair value model for the 10y (factor F3)

A regression of the 10y yield on its fundamentals, refit monthly on the trailing 10 years: **10y = constant + b1 x fed funds + b2 x core PCE inflation (12m) + b3 x real GDP growth (y/y) + b4 x Fed balance sheet as % of GDP**. The actual 10y minus the fitted value is factor F3; a negative residual (10y rich) is pressure toward higher yields. The table gives the latest refit, R-squared (flagged below 0.5), and each coefficient with its 3-year range; the chart shows actual versus fitted 10y and the R-squared over time.

In [ ]:
display(report.fair_value_summary(result))
report.fig_fair_value(result).show()

### 8.6 Data source log

Which provider delivered each factor on this run. `stale` = True means a download failed and the cached copy was used; `errors` lists failures. Expected on a machine without Bloomberg: "No module named 'xbbg'" for the Bloomberg-only factors (A6, D2, F6, G6) and for the Bloomberg-first factors that fell back to FRED (H1, H2, H4).

In [ ]:
display(result.provider_log)